In [35]:
!pip install -q faiss-cpu sentence-transformers groq flask joblib transformers torch

In [36]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/Customer_support_chatbot'
print(PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Customer_support_chatbot


In [37]:
!fuser -k 5000/tcp

5000/tcp:            31123


In [38]:
%%writefile app.py
import joblib
import pandas as pd
import re
import traceback
import faiss
from transformers import pipeline
from sentence_transformers import SentenceTransformer
from groq import Groq
from flask import Flask, request, jsonify
import os

# ============================================
# CONFIG
# ============================================
PROJECT_DIR = '/content/drive/MyDrive/Customer_support_chatbot'
import os
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")

# ============================================
# LOAD ALL 4 TRAINED MODULES
# ============================================
print("Loading language detection model...")
lang_clf = joblib.load(f'{PROJECT_DIR}/models/lang_classifier.pkl')
lang_vectorizer = joblib.load(f'{PROJECT_DIR}/models/lang_vectorizer.pkl')

print("Loading intent classifier...")
intent_clf = joblib.load(f'{PROJECT_DIR}/models/intent_classifier.pkl')
intent_vectorizer = joblib.load(f'{PROJECT_DIR}/models/intent_vectorizer.pkl')

print("Loading sentiment model...")
sentiment_pipe = pipeline(
    "text-classification",
    model=f'{PROJECT_DIR}/models/sentiment_model',
    tokenizer=f'{PROJECT_DIR}/models/sentiment_model'
)
id_to_bucket = {0: 'negative', 1: 'neutral', 2: 'positive'}

print("Loading RAG components...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
faiss_index = faiss.read_index(f'{PROJECT_DIR}/models/support_faiss.index')
support_df = pd.read_pickle(f'{PROJECT_DIR}/models/support_df.pkl')
groq_client = Groq(api_key=GROQ_API_KEY)

print("All models loaded.")

# ============================================
# RAG RETRIEVAL + GENERATION
# ============================================
def retrieve(query, top_k=3):
    query_vec = embedder.encode([query]).astype('float32')
    distances, indices = faiss_index.search(query_vec, top_k)
    results = []
    for idx in indices[0]:
        results.append({
            'instruction': support_df.iloc[idx]['instruction'],
            'response': support_df.iloc[idx]['response']
        })
    return results

def rag_answer(user_message, detected_sentiment="neutral"):
    retrieved = retrieve(user_message, top_k=3)
    context_text = "\n\n".join(
        [f"Q: {r['instruction']}\nA: {r['response']}" for r in retrieved]
    )

    system_prompt = f"""You are a helpful, professional customer support assistant for an online retailer.
Answer the customer's question using ONLY the information in the retrieved support responses below.
If the customer sounds frustrated ({detected_sentiment}), acknowledge that before answering.
If the retrieved context does not cover the question, say so honestly and offer to escalate to a human agent rather than guessing."""

    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Context:\n{context_text}\n\nCustomer question: \"{user_message}\""}
        ],
        temperature=0.3,
    )
    return response.choices[0].message.content

# ==============================================
# ROUTER — decides how each message gets handled
# ==============================================
GREETING_PATTERNS = re.compile(
    r"^\s*(hi+|hello+|hey+|good\s?(morning|afternoon|evening)|thanks|thank\s?you|"
    r"bye|goodbye|see\s?you|ok(ay)?|great|cool)\b[\s!.,]*$",
    re.IGNORECASE
)

def route_message(user_message):
    # Language detection
    lang_vec = lang_vectorizer.transform([user_message])
    detected_lang = lang_clf.predict(lang_vec)[0]

    # Sentiment detection
    sentiment_result = sentiment_pipe(user_message)[0]
    sentiment_id = int(sentiment_result['label'].split('_')[-1])
    detected_sentiment = id_to_bucket[sentiment_id]

    if GREETING_PATTERNS.match(user_message.strip()):
        return {
            'language': str(detected_lang),
            'sentiment': str(detected_sentiment),
            'intent': 'greeting',
            'response': "Hi there! How can I help you today?"
        }

    # Intent detection
    intent_vec = intent_vectorizer.transform([user_message])
    detected_intent = intent_clf.predict(intent_vec)[0]

    # Routing logic
    if detected_intent == 'complaint':
        response_text = ("I'm really sorry to hear that. I'm escalating this to a "
                          "human agent who will follow up with you shortly.")
    elif detected_intent not in [
        'order_status', 'order_management', 'billing_and_refunds', 'account_management'
    ]:
        response_text = "Thanks for reaching out! How can I help you today?"
    else:
        response_text = rag_answer(user_message, detected_sentiment)

    return {
        'language': str(detected_lang),
        'sentiment': str(detected_sentiment),
        'intent': str(detected_intent),
        'response': response_text
    }

# ============================================
# FLASK APP
# ============================================
app = Flask(__name__)

@app.route('/chat', methods=['POST'])
def chat():
    try:
        data = request.get_json()
        user_message = data.get('message', '') if data else ''
        if not user_message:
            return jsonify({'error': 'No message provided'}), 400
        result = route_message(user_message)
        return jsonify(result)
    except Exception as e:
        return jsonify({
            'error': str(e),
            'traceback': traceback.format_exc()
        }), 500

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok'})

if __name__ == '__main__':
    app.run(port=5000)

Overwriting app.py


In [40]:
from google.colab import userdata
import os
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
print("Key length:", len(os.environ['GROQ_API_KEY']))

Key length: 56


In [41]:
import os, subprocess, time, requests

# Force unbuffered stdout so the log file updates in real time
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

shutil.copy('app.py', f'{PROJECT_DIR}/app.py')

log_path = f'{PROJECT_DIR}/app_log.txt'
log_file = open(log_path, 'w')
process = subprocess.Popen(
    ['python', '-u', 'app.py'],   # -u also forces unbuffered mode
    stdout=log_file,
    stderr=subprocess.STDOUT,
    cwd=PROJECT_DIR,
    env=env
)

# Poll /health instead of a blind sleep — waits exactly as long as needed, up to a cap
max_wait = 180  # seconds
start = time.time()
ready = False
while time.time() - start < max_wait:
    if process.poll() is not None:
        print("Server process exited early! Check the log below.")
        break
    try:
        r = requests.get('http://127.0.0.1:5000/health', timeout=2)
        if r.status_code == 200:
            ready = True
            break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(2)

print("Ready:", ready, f"(waited {time.time()-start:.1f}s)")

with open(log_path) as f:
    print(f.read()[-1500:])

Ready: True (waited 22.0s)
Loading language detection model...
Loading intent classifier...
Loading sentiment model...

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 2945.02it/s]
Loading RAG components...

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11228.12it/s]
All models loaded.
 * Serving Flask app 'app'
 * Debug mode: off
 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [13/Sep/2026 20:57:15] "GET /health HTTP/1.1" 200 -



testing with some msgs

In [42]:
import requests

test_messages = [
    "Where is my order? It's been a week!",
    "This service is terrible, I want a refund now",
    "Hi there!",
    "What is your refund policy?",
]

for msg in test_messages:
    try:
        resp = requests.post('http://127.0.0.1:5000/chat', json={'message': msg}, timeout=30)
        print(f"IN:  {msg}")
        print(f"Status: {resp.status_code}")
        print(f"OUT: {resp.text}")
    except Exception as e:
        print(f"IN:  {msg}")
        print(f"REQUEST FAILED: {e}")
    print("---")

IN:  Where is my order? It's been a week!
Status: 200
OUT: {"intent":"order_status","language":"en","response":"I\u2019m sorry you\u2019re feeling frustrated\u2014let\u2019s get this sorted out quickly.  \nCould you please share your **Order Number** or **Tracking Number**? Once I have that, I can pull up the latest status and let you know exactly when your order is expected to arrive.","sentiment":"negative"}

---
IN:  This service is terrible, I want a refund now
Status: 200
OUT: {"intent":"billing_and_refunds","language":"en","response":"I\u2019m really sorry to hear you\u2019re unhappy with our service.  \nTo help you get a refund, could you please share your order number or any other details about the purchase? Once I have that information, I can walk you through the steps to initiate the refund process.","sentiment":"negative"}

---
IN:  Hi there!
Status: 200
OUT: {"intent":"billing_and_refunds","language":"en","response":"Hello! How can I help you today?","sentiment":"positive"}